# Linear response

In this tutorial you compute the linear response of Pt, the material of the base case, to a static electric field with `linear_response()`. The call gives four responses: the spin Hall conductivity, the Rashba–Edelstein tensor, the charge conductivity and the anomalous Hall conductivity. Each comes from one of five Kubo expressions, equations (1) to (5) below, which [Kubo linear response](../theory/linear-response.md) derives.

**What you will compute**

- The steps that `linear_response()` needs, and the argument that selects the response
- The Kubo expressions (1) to (5) that the call evaluates, and the observable behind each response
- The general Kubo–Bastin response of `kubo_response()`, and how its results map onto those of `linear_response()`

:::{tip}
This page is built on the case [`pt_spin_hall`](https://github.com/marcobn/PAOFLOW/tree/develop/examples/cases/pt_spin_hall): open it in PAOFLOW Studio with *Examples…* → `pt_spin_hall`, or copy it with `paoflow-gen --example pt_spin_hall`. This tutorial is not complete yet: its cells do not follow the driver of the case and have not been run on it.
:::

## The steps before `linear_response()`

`linear_response()` needs four results of earlier steps:

- the PAO Hamiltonian, from `pao_hamiltonian()`, and on a denser mesh from `interpolated_hamiltonian()`;
- the eigenvalues and eigenvectors, from `pao_eigh()`;
- the gradient of the Hamiltonian and the momenta, from `gradient_and_momenta()`;
- for the spin responses, the spin operators, from `spin_operator()`.

When `gradient_and_momenta()` or `spin_operator()` has not run, `linear_response()` calls it with its defaults.

The argument `response` selects the response, and with it the equation that is evaluated by default:

| Call | Response | Equation |
| --- | --- | --- |
| `paoflow.linear_response(response='shc')` | spin Hall conductivity tensor | (3) |
| `paoflow.linear_response(response='ree')` | Rashba–Edelstein tensor | (1) |
| `paoflow.linear_response(response='cond')` | charge conductivity tensor | (1) |
| `paoflow.linear_response(response='ahc')` | anomalous Hall conductivity tensor | (3) |

Other arguments select another of the equations (1) to (5), suited to the system, and the tensor components and energies. The cell below passes every argument except `delta` at its default value, and the next section explains each.

In [ ]:
from PAOFLOW import PAOFLOW

if __name__ == '__main__':
    paoflow = PAOFLOW.PAOFLOW(
        savedir='../pt.save/',  ##.save directory
        outputdir='output',
        verbose=1,
        npool=8,
    )
    data_controller = paoflow.data_controller
    arry, attr = data_controller.data_dicts()
    paoflow.read_atomic_proj_QE()
    paoflow.projectability()
    paoflow.pao_hamiltonian()
    # paoflow.interpolated_hamiltonian() ##usually require high mesh for convergence
    paoflow.pao_eigh()
    paoflow.gradient_and_momenta()
    paoflow.spin_operator()
    paoflow.linear_response(
        response='shc',
        gamma=0.01,
        twoD=False,
        t_odd=False,
        full_chi2=False,
        intraband=False,
        interband=False,
        s_tensor=None,
        a_tensor=None,
        eminH=-1.0,
        emaxH=1.0,
        esize=200,
    )
paoflow.finish_execution()

## Arguments of `linear_response()`

The equation numbers refer to the next section.

| Argument | Default | Meaning |
| --- | --- | --- |
| `response` | `'shc'` | `'ree'`: Rashba–Edelstein effect; `'shc'`: spin Hall conductivity; `'ahc'`: anomalous Hall conductivity; `'cond'`: charge conductivity |
| `t_odd` | `False` | For `'shc'` and `'ree'` only. `True` evaluates the time-reversal-odd part: $\chi^I$ (1) for `'shc'` and $\chi^{II}$ (3) for `'ree'`. `False` evaluates $\chi^{II}$ for `'shc'` and $\chi^I$ for `'ree'` |
| `full_chi2` | `False` | `True` evaluates $\chi^{II}$ with equation (2) instead of (3) |
| `intraband` | `False` | `True` evaluates $\chi^I$ with equation (4) instead of (1) |
| `interband` | `False` | `True` evaluates $\chi^I$ with equation (5) instead of (1) |
| `gamma` | 0.01 eV | spectral broadening $\Gamma$ |
| `delta` | 0.05 eV | regulariser of the energy denominators of equation (3) |
| `twoD` | `False` | `True` gives sheet values of a two-dimensional system, in units of $e^2/\hbar$ |
| `s_tensor` | `None` | spin Hall components as index triples (spin polarisation, spin current, applied field), for example `[[0,0,0],[2,0,1]]` |
| `a_tensor` | `None` | Rashba–Edelstein, anomalous Hall and conductivity components as index pairs (response, applied field), for example `[[0,0],[2,1]]` |
| `eminH` | $-1.0$ eV | lowest energy at which the response is computed, measured from $E_F$ |
| `emaxH` | $1.0$ eV | highest energy at which the response is computed, capped at the PAO shift |
| `esize` | 200 | number of energy steps from `eminH` to `emaxH` |

The indices 0, 1 and 2 stand for $x$, $y$ and $z$. Without `s_tensor` or `a_tensor`, `linear_response()` takes the components of an earlier `spin_Hall()` or `anomalous_Hall()` call, and stops with an error if there was none.

Set `t_odd=True` only if your system breaks time-reversal symmetry, as a magnet does. In a non-magnetic system the T-odd tensors vanish, as the table at the end of the next section shows.

## The Kubo expressions of `linear_response()`

`linear_response()` evaluates Kubo's linear response of an observable to an applied field ([The Kubo formula for a static electric field](../theory/linear-response.md#the-kubo-formula-for-a-static-electric-field)). In the constant relaxation-time approximation, the change $\delta \mathbf{A}$ of an observable that an external electric field $\mathbf{E}$ induces is

$$\delta A_i = \left(\chi^I_{ij}+\chi^{II}_{ij} \right)E_j$$

The two response tensors $\chi^{I}$ and $\chi^{II}$ differ in their parity under time reversal. Depending on the observable, one is even (T-even) and the other odd (T-odd) ([Time-reversal parity of the two terms](../theory/linear-response.md#time-reversal-parity-of-the-two-terms)).

The response tensors are:

$$\chi^{I}_{ij} = -\frac{e\hbar}{\pi}\sum_{k,n,m}\frac{\Gamma^2\text{Re}\left[\langle \psi_{nk}|\hat{A}_i|\psi_{mk}\rangle \langle \psi_{mk}|\hat{v}_j| \psi_{nk}\rangle  \right]  }
{\left[\left( E_F - E_{nk}\right)^2+\Gamma^2\right]  \left[\left( E_F - E_{mk}\right)^2+\Gamma^2\right]} \qquad(1)$$

and

$$\chi^{II}_{ij} = 2e\hbar \sum_{k, n\neq m}^{\substack{n_{\text{occ}}\\ m_{\text{unocc}}}}
\frac
{\text{Im}\left[\langle \psi_{nk}|\hat{A}_i|\psi_{mk}\rangle \langle \psi_{mk}|\hat{v}_j|\psi_{nk}\rangle \right] \left(\Gamma^2 - (E_{nk}-E_{mk})^2\right)}
{\left[(E_{nk}-E_{mk})^2+\Gamma^2 \right]^2}
\qquad(2)$$

with the symbols

| Symbol | Meaning |
| --- | --- |
| $n$, $m$ | band indices |
| $k$ | crystal momentum |
| $E_F$ | Fermi energy, or the energy at which you evaluate the response |
| $\Gamma$ | spectral broadening, related to the relaxation time $\tau$ as $\Gamma = \frac{\hbar}{2\tau}$ |
| $\hat{A}$ | operator of the observable $\mathbf{A}$ (table below) |
| $\hat{v}$ | velocity operator, the $k$-gradient of the Hamiltonian divided by $\hbar$ |

In the limit $\Gamma \rightarrow 0$, equation (2) reduces to

$$\chi^{II}_{ij} \approx 2e\hbar \sum_{k, n\neq m}^{\substack{n_{\text{occ}}\\ m_{\text{unocc}}}}\frac
{\text{Im}\left[\langle \psi_{nk}|\hat{A}_i|\psi_{mk}\rangle \langle \psi_{mk}|\hat{v}_j|\psi_{nk}\rangle \right]}
{(E_{nk}-E_{mk})^2}
\qquad(3)$$

For small $\Gamma$, equation (1) splits into an intraband and an interband part,

$$\chi^I_{\text{intra}} = -\frac{1}{V}\frac{e\hbar}{2\Gamma}\sum_{k,n} \text{Re}\left[\langle \psi_{nk}|\hat{A}_i|\psi_{nk}\rangle \langle \psi_{nk}|\hat{v}_j|\psi_{nk}\rangle \right]\delta E_{nk}
\qquad(4)$$

and

$$\chi^{I}_{\text{inter}} = -\frac{e\hbar}{V} \frac
{\text{Re} \left[\langle \psi_{nk}|\hat{A}_i|\psi_{mk}\rangle \langle \psi_{mk}|\hat{A}_i|\psi_{nk}\rangle \right] (E_{nk}-E_{mk})\Gamma}
{\left[(E_{nk}-E_{mk})^2+\Gamma^2\right]^2}
\qquad(5)$$

## Notes on operator $\hat{A}$

The observable decides what each tensor describes. A T-odd tensor needs broken time-reversal symmetry, so it vanishes in a non-magnetic system; a T-even tensor exists in magnetic and non-magnetic systems alike.

| $\hat{A}$ | $\chi^I$ | $\chi^{II}$ |
| --- | --- | --- |
| $\hat{v}$, velocity | electrical conductivity | anomalous Hall effect (AHE) |
| $\hat{S}$, spin | T-even: Rashba–Edelstein effect (REE) | T-odd: magnetic Rashba–Edelstein effect |
| $\hat{j} = \frac{1}{2}\{\hat{S}_k, \hat{v}_j\}$, spin current | T-odd: magnetic spin Hall conductivity | T-even: spin Hall conductivity |

## References
- [Persistent spin textures, altermagnetism and charge-to-spin conversion in metallic chiral crystals TM3X6](https://doi.org/10.1038/s44306-025-00109-9)
- [Spin-orbit torques in Co/Pt(111) and Mn/W(001) magnetic bilayers from first principles](https://doi.org/10.1103/PhysRevB.90.174423)
- [Efficient electrical spin splitter based on nonrelativistic collinear antiferromagnetism](https://doi.org/10.1103/PhysRevLett.126.127701)
- [Spin-polarized current in noncollinear antiferromagnets](https://doi.org/10.1103/PhysRevLett.119.187204)
- [Non-relativistic torque and Edelstein effect in non-collinear magnets](https://doi.org/10.1038/s41467-024-51565-6)
- [Intraband and interband spin-orbit torques in noncentrosymmetric ferromagnets](https://doi.org/10.1103/PhysRevB.91.134402)

## The general route: `kubo_response()`

`kubo_response()` evaluates the Kubo–Bastin response at constant broadening $\eta = \hbar/2\tau$ exactly, for any observable and any term. Every equation above is a limit of this one formula. Its arguments select:

| Argument | Choices |
| --- | --- |
| observable (first argument) | `charge_current`, `spin_current`, `orbital_current`, `spin_density`, `orbital_density`, `spin_torque`, or `custom`, a matrix you give |
| `terms` | `total` (surface + sea); `surface` (equation (1) at $T = 0$); `sea` (the Fermi sea at finite $\eta$, exact); `surface_intraband` and `surface_interband`; `intrinsic` (the $\eta \to 0$ Berry-curvature form) |
| `temperature` | any, by an exact thermal convolution of the $T = 0$ kernels |
| components (second argument) | letters, the field direction last: `'xy'`, `'zxy'` (spin $z$, current $x$, field $y$), `'yx'` (spin $y$, field $x$) |

`kubo_response()` writes one file per component, `kubo_<observable>_<component>.dat`, which holds $\mu$ and one column per term.

`kubo_optical_response()` gives the frequency-dependent (optical) response of the same observables. It uses the Kubo–Greenwood formula, interband transitions plus a Drude term, and writes the files `kubo_optical_<observable>_<component>.dat` with the real and imaginary parts.

Each call of `linear_response()` has an equivalent call of `kubo_response()`:

| `linear_response` call | `kubo_response` equivalent |
|---|---|
| `response='cond'` (Eq. 1) | `'charge_current'`, `terms=('surface',)` |
| `response='shc'` (Eq. 3) | `'spin_current'`, `terms=('intrinsic',)`, `regulariser=0.05` |
| `response='ahc'` (Eq. 3) | `'charge_current'`, `terms=('intrinsic',)`, `regulariser=0.05`, opposite sign (see below) |
| `response='ree'` (Eq. 1) | `'spin_density'`, `terms=('surface',)`, opposite sign (see below) |
| `response='ree', t_odd=True` (Eq. 3) | `'spin_density'`, `terms=('intrinsic',)`, `regulariser=0.05`, opposite sign |

`kubo_response()` uses the Kubo formula with the electron charge $q = -|e|$: $q^2$ for the charge current (Yao et al. 2004), and $q$ for every other observable (the spin Hall convention of Guo et al. 2008). Its spin Hall values agree with `spin_Hall()`, $+2012$ $(\hbar/e)$ S/cm for Pt. Two other outputs have the opposite sign, for different reasons:

| Output | Why its sign is opposite to that of `kubo_response()` |
| --- | --- |
| the Rashba–Edelstein tensors of `linear_response()` | they take `dHksp`, which is $-\hbar v$, as the velocity |
| `anomalous_Hall()` and equation (3) | they report $\sigma_{xy}$ for the magnetic moment along the spin axis of the run, whereas `kubo_response()` reports the computed state. A positive `starting_magnetization` in QE puts the spins along $+z$, so the moment points along $-z$ |

[Kubo–Bastin linear response](../../internals/kubo-bastin.md) gives the details, the closed forms and the validation.

In [ ]:
# Continuing the Pt run above (spin-orbit, after gradient_and_momenta):
shc = paoflow.kubo_response(
    'spin_current',
    ['zxy'],
    terms=('total', 'surface', 'sea', 'intrinsic'),
    eta=0.01,
    emin=-1.0,
    emax=1.0,
    ne=201,
)
ree = paoflow.kubo_response('spin_density', ['yx'], terms=('total',), eta=0.01, temperature=300.0)
print(shc['z_xy']['total'][100], ree['yx']['total'][100])
# Optical (Kubo-Greenwood) conductivity and AC spin Hall at the Fermi level:
optics = paoflow.kubo_optical_response(
    'charge_current', ['xx', 'xy'], eta=0.05, omega_max=6.0, nomega=301
)
ac_shc = paoflow.kubo_optical_response('spin_current', ['zxy'], eta=0.05, omega_max=6.0, nomega=301)